# Single-Stock Risk Report

Parameterized risk report built on `ohlc_hw_cleaned`. Run it on its own, or through `03_run_all_stocks.py`, which uses papermill to override the tagged parameters cell below.

In [ ]:
ticker = "AAPL"
start_date = "2024-01-01"
end_date = "2024-12-31"
db_path = "data/data.db"

In [ ]:
import sqlite3
from contextlib import closing
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scrapbook as sb
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.ticker import PercentFormatter

TRADING_DAYS = 252
MIN_PRICES = 21

GREEN, RED, GRAY, BLUE = "#2e7d32", "#c62828", "#9e9e9e", "#1565c0"

## 1. Load data

In [ ]:
def load_prices(db_path, ticker, start_date, end_date):
    """Read one ticker's cleaned rows for [start_date, end_date] from the database (read-only)."""
    path = Path(db_path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"database not found: {path}")
    start, end = pd.Timestamp(start_date), pd.Timestamp(end_date)
    if start > end:
        raise ValueError(f"start_date {start_date} is after end_date {end_date}")

    with closing(sqlite3.connect(f"{path.as_uri()}?mode=ro", uri=True)) as conn:
        if conn.execute("SELECT 1 FROM ohlc_hw_cleaned WHERE ticker = ? LIMIT 1", (ticker,)).fetchone() is None:
            raise ValueError(f"unknown ticker: {ticker!r}")
        df = pd.read_sql_query(
            "SELECT ts, open, high, low, close, adj_close, volume FROM ohlc_hw_cleaned "
            "WHERE ticker = ? AND ts BETWEEN ? AND ? ORDER BY ts",
            conn,
            params=(ticker, f"{start:%Y-%m-%d}", f"{end:%Y-%m-%d}"),
            parse_dates=["ts"],
        )

    if len(df) < MIN_PRICES:
        raise ValueError(
            f"{ticker}: only {len(df)} prices between {start_date} and {end_date}; "
            f"the report needs at least {MIN_PRICES} (20 returns)"
        )
    return df.set_index("ts")


prices = load_prices(db_path, ticker, start_date, end_date)

price_cols = ["open", "high", "low", "close", "adj_close"]
bad_rows = (
    (prices[price_cols] <= 0).any(axis=1)
    | ~np.isfinite(prices[price_cols]).all(axis=1)
    | (prices["low"] > prices[["open", "close"]].min(axis=1))
    | (prices["high"] < prices[["open", "close"]].max(axis=1))
)
assert prices.index.is_unique, "duplicate dates in cleaned data"
assert not bad_rows.any(), f"{bad_rows.sum()} rows fail the cleaning rules"

actual_start, actual_end = prices.index[0], prices.index[-1]
print(f"{ticker} from {actual_start:%Y-%m-%d} to {actual_end:%Y-%m-%d}: {len(prices)} trading days")
print(f"Requested window: {start_date} to {end_date}")

coverage_notes = []
if (actual_start - pd.Timestamp(start_date)).days > 5:
    coverage_notes.append(f"data starts {actual_start:%Y-%m-%d}, after the requested start")
if (pd.Timestamp(end_date) - actual_end).days > 5:
    coverage_notes.append(f"data ends {actual_end:%Y-%m-%d}, before the requested end")
for note in coverage_notes:
    print(f"Coverage note: {note}. Metrics describe the observed period only.")

pd.concat([prices.head(), prices.tail()])

## 2. Risk metrics

Returns and drawdown use **adjusted close**, and all return statistics use daily **log** returns.

In [ ]:
def calculate_returns(prices: pd.Series) -> pd.Series:
    """Calculate daily log returns. The first value is NaN."""
    return np.log(prices / prices.shift(1))


def calculate_volatility(returns: pd.Series, window: int = 20) -> pd.Series:
    """Calculate annualized rolling volatility (sample std of `window` returns x sqrt(252))."""
    return returns.rolling(window).std() * np.sqrt(TRADING_DAYS)


def calculate_max_drawdown(prices: pd.Series) -> pd.Series:
    """Return per-date drawdown from the running price peak. Maximum drawdown is `.min()` of this."""
    return prices / prices.cummax() - 1


def calculate_sharpe_ratio(returns: pd.Series, risk_free_rate: float = 0.02) -> float:
    """Calculate annualized Sharpe ratio; NaN if there are too few returns or zero volatility."""
    r = returns.dropna()
    if len(r) < 2:
        return np.nan
    sd = r.std()
    # constant returns give a std of ~1e-18 rather than exactly 0
    if not np.isfinite(sd) or sd < 1e-12:
        return np.nan
    return (r.mean() * TRADING_DAYS - risk_free_rate) / (sd * np.sqrt(TRADING_DAYS))

In [ ]:
toy = pd.Series([80.0, 100.0, 70.0, 85.0])
toy_dd = calculate_max_drawdown(toy)
assert np.isclose(toy_dd.min(), -0.30) and np.isclose(toy_dd.iloc[-1], -0.15)
assert np.isclose(calculate_returns(toy).iloc[1], np.log(100 / 80))
assert np.isnan(calculate_sharpe_ratio(pd.Series([0.01] * 30)))
assert np.isnan(calculate_sharpe_ratio(pd.Series([np.nan, 0.01])))
print("metric checks passed")

In [ ]:
adj = prices["adj_close"]
log_ret = calculate_returns(adj)
roll_vol = calculate_volatility(log_ret)
drawdown = calculate_max_drawdown(adj)
ma50 = adj.rolling(50).mean()

r = log_ret.dropna()
n = len(r)
up, down = r[r > 0], r[r < 0]
mean_roll_vol = roll_vol.mean()

metrics = {
    "total_return": adj.iloc[-1] / adj.iloc[0] - 1,
    "annualized_return": np.exp(r.sum() * TRADING_DAYS / n) - 1,
    "annualized_volatility": r.std() * np.sqrt(TRADING_DAYS),
    "sharpe_ratio": calculate_sharpe_ratio(log_ret),
    "max_drawdown": drawdown.min(),
    "best_day": r.max(),
    "worst_day": r.min(),
    "positive_days": (r > 0).sum() / n,
    "avg_up_day": up.mean() if len(up) else np.nan,
    "avg_down_day": down.mean() if len(down) else np.nan,
    "avg_daily_return": r.mean(),
    "avg_volatility": mean_roll_vol,
}

pd.DataFrame({"adj_close": adj, "log_return": log_ret, "vol_20d": roll_vol, "drawdown": drawdown}).tail(10)

## 3. Charts

In [ ]:
def plot_price(ax):
    ax.plot(adj.index, ma50, color=BLUE, lw=1.8, ls="--", label="50-day MA")
    x = mdates.date2num(adj.index.to_pydatetime())
    pts = np.column_stack([x, adj.to_numpy()])
    segments = np.stack([pts[:-1], pts[1:]], axis=1)
    seg_vol = roll_vol.to_numpy()[1:]
    colors = np.where(np.isnan(seg_vol), GRAY, np.where(seg_vol > mean_roll_vol, RED, GREEN))
    ax.add_collection(LineCollection(segments, colors=colors, linewidths=2))
    ax.autoscale_view()

    handles = [
        Line2D([], [], color=GREEN, lw=2, label="20d vol <= mean"),
        Line2D([], [], color=RED, lw=2, label="20d vol > mean"),
        Line2D([], [], color=GRAY, lw=2, label="vol warm-up (n/a)"),
        Line2D([], [], color=BLUE, lw=1.8, ls="--", label="50-day MA"),
    ]
    ax.legend(handles=handles, loc="best", fontsize=9)
    if ma50.notna().sum() == 0:
        ax.text(0.02, 0.04, f"50-day MA unavailable: only {len(adj)} prices", transform=ax.transAxes,
                fontsize=9, color=BLUE, bbox=dict(fc="white", ec=BLUE, alpha=0.8))
    ax.set_title(f"{ticker} Price and 50-day MA")
    ax.set_ylabel("Adjusted close ($)")


def plot_returns(ax):
    colors = np.select([r > 0, r < 0], [GREEN, RED], GRAY)
    ax.bar(r.index, r.to_numpy(), color=colors, width=1.0)
    ax.axhline(0, color="black", lw=0.8)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(f"{ticker} Daily Returns")
    ax.set_ylabel("Daily log return")


def plot_volatility(ax):
    ax.plot(roll_vol.index, roll_vol, color="black", lw=1.5, label="20-day volatility")
    ax.axhline(mean_roll_vol, color=BLUE, ls="--", lw=1.2, label=f"Mean ({mean_roll_vol:.1%})")
    ax.fill_between(roll_vol.index, roll_vol, mean_roll_vol, where=roll_vol > mean_roll_vol,
                    color=RED, alpha=0.25, interpolate=True, label="High vol")
    ax.fill_between(roll_vol.index, roll_vol, mean_roll_vol, where=roll_vol <= mean_roll_vol,
                    color=GREEN, alpha=0.25, interpolate=True, label="Low vol")
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=1))
    ax.legend(loc="best", fontsize=9)
    ax.set_title(f"{ticker} Rolling Volatility (20-day)")
    ax.set_ylabel("Annualized volatility")


def plot_drawdown(ax):
    for frac, alpha in [(1.0, 0.18), (0.75, 0.18), (0.5, 0.18), (0.25, 0.18)]:
        ax.fill_between(drawdown.index, drawdown * frac, 0, color=RED, alpha=alpha, lw=0)
    ax.plot(drawdown.index, drawdown, color=RED, lw=1.2)
    trough = drawdown.idxmin()
    ax.scatter([trough], [drawdown.min()], color="black", zorder=5, s=40)
    ax.annotate(f"Max DD {drawdown.min():.1%}\n{trough:%Y-%m-%d}", (trough, drawdown.min()),
                xytext=(10, 10), textcoords="offset points", fontsize=9,
                bbox=dict(fc="white", ec="black", alpha=0.8))
    ax.set_ylim(min(drawdown.min() * 1.25, -0.01), 0.005)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(f"{ticker} Maximum Drawdown")
    ax.set_ylabel("Drawdown from peak")


fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for plot, ax in zip([plot_price, plot_returns, plot_volatility, plot_drawdown], axes.flat):
    plot(ax)
    ax.grid(True, alpha=0.35)
    ax.set_xlim(adj.index[0] - pd.Timedelta(days=3), adj.index[-1] + pd.Timedelta(days=3))
    ax.set_xlabel("Date")
    locator = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

fig.suptitle(f"{ticker} Risk Analysis Report - {start_date} to {end_date}", fontsize=18, fontweight="bold")
if coverage_notes:
    fig.text(0.5, 0.945, f"Data available {actual_start:%Y-%m-%d} to {actual_end:%Y-%m-%d} ({len(prices)} trading days)",
             ha="center", fontsize=11, color="#b71c1c")
fig.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

Line and shading colors compare each day's 20-day volatility with its average over this report, so the coloring is retrospective. Gray marks the first 20 days, where no rolling estimate exists yet.

## 4. Summary

In [ ]:
def fmt_pct(x, signed=True):
    if x is None or not np.isfinite(x):
        return "N/A"
    return f"{x:+.2%}" if signed else f"{x:.2%}"


summary = pd.DataFrame(
    [
        ("Total Return", fmt_pct(metrics["total_return"])),
        ("Annualized Return", fmt_pct(metrics["annualized_return"])),
        ("Annualized Volatility", fmt_pct(metrics["annualized_volatility"], signed=False)),
        ("Sharpe Ratio", "N/A" if np.isnan(metrics["sharpe_ratio"]) else f"{metrics['sharpe_ratio']:.2f}"),
        ("Maximum Drawdown", fmt_pct(metrics["max_drawdown"])),
        ("Best Day", fmt_pct(metrics["best_day"])),
        ("Worst Day", fmt_pct(metrics["worst_day"])),
        ("Positive Days", fmt_pct(metrics["positive_days"], signed=False)),
        ("Average Up Day", fmt_pct(metrics["avg_up_day"])),
        ("Average Down Day", fmt_pct(metrics["avg_down_day"])),
    ],
    columns=["Metric", "Value"],
).set_index("Metric")

print(f"{ticker}: {actual_start:%Y-%m-%d} to {actual_end:%Y-%m-%d}, {len(prices)} prices / {n} returns")
summary

In [ ]:
def to_float(x):
    x = float(x)
    return x if np.isfinite(x) else None


sb.glue("metrics", {
    "ticker": ticker,
    **{k: to_float(v) for k, v in metrics.items()},
    "requested_start": start_date,
    "requested_end": end_date,
    "actual_start": f"{actual_start:%Y-%m-%d}",
    "actual_end": f"{actual_end:%Y-%m-%d}",
    "n_prices": int(len(prices)),
})